# Advanced Python `object` class: 28 problems and worked solutions

**Scope:** Python's root `object` type; the `type` / instance distinction; inheritance and method resolution; identity, equality, hashing; `__repr__`, `__str__`, `__format__`; `__new__`, `__init__`, `super()`; attribute and special-method lookup; descriptors, slots, subclass hooks and dynamic classes.

**Based on the supplied lesson** on the `object` class and expanded with new advanced exercises and carefully tested solutions. Examples deliberately use portable assertions rather than process-specific `id()` values or memory addresses.

**How to study:** For each numbered problem, write a prediction or implementation before running its solution. Execute **Kernel → Restart & Run All** for a clean run. All exercises use only the Python standard library; Python 3.10+ is recommended. The assertions act as executable self-checks. Each solution uses fresh class names to minimize interference across exercises.

**Important precision:** For two distinct objects, `object.__eq__(a, b)` can return `NotImplemented`; the full `a == b` operation consults both operands and ultimately falls back to identity. `id()` is a process-specific identity token, not a value to hard-code.

In [1]:
import copy
import math
import sys
import types
import weakref
from reprlib import recursive_repr

print(f"Python runtime: {sys.version_info.major}.{sys.version_info.minor}")
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer for this notebook."

Python runtime: 3.13


---
# Part I — Object identity, type relationships and inheritance

## Problem 01 — Reconstruct the object/type graph · Advanced

**Learning goal:** Distinguish class objects, instances, superclass relationships, and metaclasses.

**Task:** Show that a user-defined class inherits from `object`, that instances and classes are different objects, and that even `type` is an instance of itself. Use assertions, not an ASCII diagram.

**Hint:** Check `__bases__`, `type(...)`, `isinstance(...)`, and `issubclass(...)` separately.

**Solution (run the next cell):**

In [2]:
class P01Node:
    pass

node = P01Node()
assert type(node) is P01Node
assert isinstance(node, P01Node)
assert isinstance(node, object)
assert issubclass(P01Node, object)
assert P01Node.__bases__ == (object,)
assert type(P01Node) is type
assert isinstance(P01Node, object)
assert type(type) is type
assert issubclass(type, object)
assert type(object) is type
print('class → metaclass:', P01Node.__name__, '→', type(P01Node).__name__)
print('instance → class:', type(node).__name__, '→', type(P01Node).__name__)

class → metaclass: P01Node → type
instance → class: P01Node → type


**Why it works / takeaway:** An instance's type is its class; a class is itself an object, normally created by the metaclass `type`. `object` is the ultimate ordinary superclass, whereas `type` creates classes.

## Problem 02 — Inspect modules, functions and their types · Advanced

**Learning goal:** Differentiate an object from the class that creates its kind.

**Task:** Verify the standard-library module type, function type, and their inheritance from `object`. Determine whether a module object is itself a class.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [3]:
def p02_function(x):
    return x * 2

assert type(math) is types.ModuleType
assert isinstance(math, object)
assert issubclass(types.ModuleType, object)
assert type(p02_function) is types.FunctionType
assert issubclass(types.FunctionType, object)
assert isinstance(p02_function, object)
assert not isinstance(math, type)
assert isinstance(types.ModuleType, type)
assert p02_function(21) == 42
print('module object:', type(math).__name__)
print('function object:', type(p02_function).__name__)

module object: module
function object: function


**Why it works / takeaway:** Modules and functions are ordinary objects that have particular types. A module *instance* is not a class, but `types.ModuleType` is a class.

## Problem 03 — Find the actual source of an inherited method · Advanced

**Learning goal:** Use the method-resolution order and class namespaces correctly.

**Task:** Create a three-class chain, leave an attribute inherited, override a second method, and prove which class owns each implementation without comparing numeric `id()` output.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [4]:
class P03Base:
    def identify(self):
        return 'base'

    def describe(self):
        return 'base description'

class P03Middle(P03Base):
    def describe(self):
        return 'middle description'

class P03Leaf(P03Middle):
    pass

obj = P03Leaf()
assert P03Leaf.__mro__ == (P03Leaf, P03Middle, P03Base, object)
assert 'identify' not in vars(P03Leaf)
assert 'identify' not in vars(P03Middle)
assert 'identify' in vars(P03Base)
assert P03Leaf.identify is P03Base.identify
assert P03Leaf.describe is P03Middle.describe
assert obj.identify() == 'base'
assert obj.describe() == 'middle description'
print('MRO:', ' -> '.join(cls.__name__ for cls in P03Leaf.__mro__))

MRO: P03Leaf -> P03Middle -> P03Base -> object


**Why it works / takeaway:** `vars(SomeClass)` shows only the class's *own* namespace. Attribute access follows the MRO and returns inherited methods even when they are absent from the leaf's dictionary.

## Problem 04 — Separate `is`, `==`, and direct `object.__eq__` · Advanced

**Learning goal:** Understand identity fallback and `NotImplemented` precisely.

**Task:** Compare two fresh instances, alias one of them, and call the base implementation directly. Explain why `NotImplemented` must not be converted to a Boolean.

**Hint:** Call `object.__eq__` directly and then compare the result with `==`.

**Solution (run the next cell):**

In [5]:
class P04Token:
    pass

a = P04Token()
b = P04Token()
alias = a
assert a is alias
assert a == alias
assert a is not b
assert a != b
assert object.__eq__(a, alias) is True
assert object.__eq__(a, b) is NotImplemented
assert (a == b) is False
assert (a == alias) is True
print('distinct direct method:', object.__eq__(a, b))
print('distinct == operator:', a == b)

distinct direct method: NotImplemented
distinct == operator: False


**Why it works / takeaway:** For distinct objects, the base comparison declines responsibility with `NotImplemented`. Python's equality machinery can try the other operand and, if neither handles it, falls back to identity. `NotImplemented` is a protocol sentinel, not a Boolean result.

## Problem 05 — Engineer useful `repr` and `str` · Advanced

**Learning goal:** Provide debugging and user-facing representations with safe escaping.

**Task:** Build a record whose `repr` identifies its class and contains an escaped string, while `str` is concise. Test difficult quote and newline characters, and compare this behavior with the default fallback.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [6]:
class P05Record:
    def __init__(self, name, count):
        self.name = name
        self.count = count

    def __repr__(self):
        return f'{type(self).__name__}(name={self.name!r}, count={self.count!r})'

    def __str__(self):
        return f'{self.name}: {self.count}'

r = P05Record("Ada's" + chr(10) + "notes", 3)
assert repr(r) == f'P05Record(name={r.name!r}, count=3)'
assert str(r) == r.name + ': 3'
assert f'{r!r}' == repr(r)
assert f'{r!s}' == str(r)

class P05Fallback:
    def __repr__(self):
        return 'fallback-repr'

assert str(P05Fallback()) == 'fallback-repr'
print('repr:', repr(r))
print('str: ', str(r))

repr: P05Record(name="Ada's\nnotes", count=3)
str:  Ada's
notes: 3


**Why it works / takeaway:** `!r` escapes embedded strings correctly. `repr` is useful for debugging, while `str` can be human-friendly. When a class only supplies `__repr__`, the inherited `object.__str__` invokes that representation.

## Problem 06 — Implement formatting without breaking empty specs · Advanced

**Learning goal:** Distinguish `str`, `repr`, `format`, and `__format__`.

**Task:** Implement a measurement that accepts empty and `.2f` format specifications, rejects unsupported specs clearly, and inspect the default format behavior of `object`.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [7]:
class P06Measurement:
    def __init__(self, value):
        self.value = float(value)

    def __str__(self):
        return f'{self.value} m'

    def __format__(self, spec):
        if spec == '':
            return str(self)
        if spec == '.2f':
            return f'{self.value:.2f} m'
        raise ValueError(f'unsupported format spec: {spec!r}')

m = P06Measurement(1.234)
assert format(m) == '1.234 m'
assert f'{m:.2f}' == '1.23 m'
plain = object()
assert format(plain, '') == str(plain)
try:
    format(plain, 'anything')
except TypeError:
    pass
else:
    raise AssertionError('object formatting must reject nonempty spec')
try:
    format(m, '.5g')
except ValueError as exc:
    assert 'unsupported' in str(exc)
else:
    raise AssertionError('unsupported custom specification should fail')
print(format(m), '|', format(m, '.2f'))

1.234 m | 1.23 m


**Why it works / takeaway:** A custom `__format__` handles the exact format language you promise. `object.__format__` accepts an empty spec and rejects nonempty specs; use the same instance when comparing its default `str` and `format` output.

## Problem 07 — Design symmetric equality across two types · Advanced

**Learning goal:** Return `NotImplemented` instead of incorrectly returning `False`.

**Task:** Compare an amount object with a compatible foreign type. Implement matching equality on the foreign type and show that the Python operator can delegate to it.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [8]:
class P07Amount:
    def __init__(self, cents):
        self.cents = cents

    def __eq__(self, other):
        if isinstance(other, P07Amount):
            return self.cents == other.cents
        return NotImplemented

class P07Receipt:
    def __init__(self, total_cents):
        self.total_cents = total_cents

    def __eq__(self, other):
        if isinstance(other, P07Amount):
            return self.total_cents == other.cents
        return NotImplemented

amount = P07Amount(125)
receipt = P07Receipt(125)
assert P07Amount.__eq__(amount, receipt) is NotImplemented
assert amount == receipt
assert receipt == amount
assert amount != P07Amount(200)
assert amount != object()
assert P07Amount.__hash__ is None
print('cross-type equality:', amount == receipt)

cross-type equality: True


**Why it works / takeaway:** If an operation does not support the other type, return `NotImplemented` so Python can try the other object's reflected comparison. Since both custom classes define equality but not hash, both are unhashable by default.

## Problem 08 — Predict subclass-priority equality dispatch · Expert

**Learning goal:** Understand the special dispatch priority of a subclass operand.

**Task:** Use a call log to demonstrate which operand gets the first equality call when the right-hand operand has a strict subclass type.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [9]:
p08_calls = []

class P08Base:
    def __eq__(self, other):
        p08_calls.append('base')
        return NotImplemented

class P08Child(P08Base):
    def __eq__(self, other):
        p08_calls.append('child')
        return isinstance(other, P08Base)

base = P08Base()
child = P08Child()
assert base == child
assert p08_calls == ['child']
p08_calls.clear()
assert child == base
assert p08_calls == ['child']
print('dispatch log:', p08_calls)

dispatch log: ['child']


**Why it works / takeaway:** When the right operand's type is a strict subclass and implements a different equality method, Python gives it priority. This is more subtle than simply saying equality always begins on the left.

---
# Part II — Equality, hash contracts and object construction

## Problem 09 — Diagnose an automatically unhashable class · Advanced

**Learning goal:** Apply Python’s `__eq__` / `__hash__` contract.

**Task:** Create a mutable value class with equality and no custom hash. Verify that hashing fails and discover the exact value assigned to its `__hash__` attribute.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [10]:
class P09MutableValue:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if isinstance(other, P09MutableValue):
            return self.value == other.value
        return NotImplemented

x = P09MutableValue(5)
y = P09MutableValue(5)
assert x == y and x is not y
assert P09MutableValue.__hash__ is None
try:
    hash(x)
except TypeError as exc:
    assert 'unhashable' in str(exc)
else:
    raise AssertionError('expected TypeError for mutable value object')
print('hash status:', P09MutableValue.__hash__)

hash status: None


**Why it works / takeaway:** If `__eq__` is defined without `__hash__`, Python assigns `__hash__ = None`. This protects dictionaries and sets from many bugs caused by mutable equality keys.

## Problem 10 — Create a practical immutable, hashable key · Expert

**Learning goal:** Keep equality, hashing, and mutation behavior consistent.

**Task:** Implement an immutable 2D key with `__slots__`, constructor validation, a value-based hash, and a helpful `repr`. Demonstrate interchangeable dictionary lookups.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [11]:
class P10Point:
    __slots__ = ('_x', '_y', '_sealed')

    def __init__(self, x, y):
        if not all(type(v) is int for v in (x, y)):
            raise TypeError('x and y must be integers, not bools')
        object.__setattr__(self, '_x', x)
        object.__setattr__(self, '_y', y)
        object.__setattr__(self, '_sealed', True)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __setattr__(self, name, value):
        raise AttributeError('P10Point is immutable')

    def __delattr__(self, name):
        raise AttributeError('P10Point is immutable')

    def __eq__(self, other):
        if type(other) is type(self):
            return (self.x, self.y) == (other.x, other.y)
        return NotImplemented

    def __hash__(self):
        return hash((type(self), self.x, self.y))

    def __repr__(self):
        return f'P10Point({self.x!r}, {self.y!r})'

p = P10Point(2, 7)
q = P10Point(2, 7)
assert p is not q and p == q and hash(p) == hash(q)
assert {p: 'stored'}[q] == 'stored'
assert len({p, q}) == 1
assert repr(p) == 'P10Point(2, 7)'
for mutation in (lambda: setattr(p, '_x', 100), lambda: delattr(p, '_y')):
    try:
        mutation()
    except AttributeError:
        pass
    else:
        raise AssertionError('ordinary mutation should be blocked')
print('dictionary lookup via equal key:', {p: 'stored'}[q])

dictionary lookup via equal key: stored


**Why it works / takeaway:** A value object can be a hash key when its equality-relevant fields remain unchanged. The tuple-based hash agrees with the exact-type equality policy. Custom `__setattr__` prevents *ordinary* reassignment but is not a security boundary against deliberate low-level mutation.

## Problem 11 — Reproduce and explain a broken dictionary key · Expert

**Learning goal:** Recognize why mutable values cannot safely determine a hash.

**Task:** Intentionally implement a flawed class that hashes mutable state; mutate a key and demonstrate a lost dictionary lookup without assuming any particular hash numbers.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [12]:
class P11BadKey:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if isinstance(other, P11BadKey):
            return self.value == other.value
        return NotImplemented

    def __hash__(self):
        return hash(self.value)

key = P11BadKey(1)
mapping = {key: 'payload'}
assert mapping[key] == 'payload'
old_hash = hash(key)
key.value = 2
assert hash(key) != old_hash
assert mapping.get(key) is None, 'lookup is lost after the key hash changes'
print('hash before/after:', old_hash, hash(key))
print('retrieval after mutation:', mapping.get(key))

hash before/after: 1 2
retrieval after mutation: None


**Why it works / takeaway:** This deliberately unsafe design violates the requirement that a key's hash stay fixed during its lifetime in a dict. A correct design makes keys immutable (Problem 10), or uses identity-based equality and hashing. Here, Python's integer hashes of 1 and 2 are different, so the lookup failure is deterministic for this example; never design keys whose hash can change.

## Problem 12 — Subclass an immutable built-in with `__new__` · Expert

**Learning goal:** Know when initialization must happen before `__init__`.

**Task:** Create an immutable tuple subclass that always stores exactly two positive integers. Show that validation belongs in `__new__`, and confirm its tuple behavior.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [13]:
class P12PositivePair(tuple):
    def __new__(cls, left, right):
        if not (type(left) is int and type(right) is int):
            raise TypeError('both components must be integers')
        if left <= 0 or right <= 0:
            raise ValueError('both components must be positive')
        return super().__new__(cls, (left, right))

pair = P12PositivePair(3, 4)
assert isinstance(pair, tuple)
assert type(pair) is P12PositivePair
assert tuple(pair) == (3, 4)
assert hash(pair) == hash((3, 4))
try:
    P12PositivePair(1, -2)
except ValueError:
    pass
else:
    raise AssertionError('invalid pair was accepted')
print('validated tuple subclass:', pair)

validated tuple subclass: (3, 4)


**Why it works / takeaway:** Immutable built-ins such as `tuple` must receive their contents at allocation time in `__new__`; their contents cannot be repaired in `__init__` after creation.

## Problem 13 — Trace `__new__` → `__init__` correctly · Expert

**Learning goal:** Separate instance allocation from initialization and validate failures.

**Task:** Log construction order, validate inputs, and prove that `__init__` must return `None`, whereas `__new__` returns the instance.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [14]:
p13_events = []

class P13Account:
    def __new__(cls, owner):
        p13_events.append(('new', owner))
        instance = super().__new__(cls)
        assert isinstance(instance, cls)
        return instance

    def __init__(self, owner):
        p13_events.append(('init', owner))
        if not isinstance(owner, str) or not owner.strip():
            raise ValueError('nonempty owner required')
        self.owner = owner

acc = P13Account('Ada')
assert p13_events == [('new', 'Ada'), ('init', 'Ada')]
assert acc.owner == 'Ada'

class P13InvalidInit:
    def __init__(self):
        return 'wrong'  # intentionally invalid

try:
    P13InvalidInit()
except TypeError as exc:
    assert '__init__' in str(exc)
else:
    raise AssertionError('returning from __init__ should fail')
print('construction events:', p13_events)

construction events: [('new', 'Ada'), ('init', 'Ada')]


**Why it works / takeaway:** `__new__` allocates/returns an object; if it returns an instance of the requested class, `__init__` initializes it and must return `None`. `object.__new__` and `object.__init__` provide default hooks for classes that do not override them.

## Problem 14 — Write cooperative mixin constructors · Expert

**Learning goal:** Consume only your own keyword arguments and forward the rest.

**Task:** Combine a `name` mixin and an `audit_tag` mixin using `super()` without sending extra keywords to `object.__init__`.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [15]:
class P14NamedMixin:
    def __init__(self, *, name, **kwargs):
        self.name = name
        super().__init__(**kwargs)

class P14AuditMixin:
    def __init__(self, *, audit_tag, **kwargs):
        self.audit_tag = audit_tag
        super().__init__(**kwargs)

class P14Entry(P14NamedMixin, P14AuditMixin):
    pass

entry = P14Entry(name='shipment', audit_tag='A-17')
assert entry.name == 'shipment'
assert entry.audit_tag == 'A-17'
assert P14Entry.__mro__ == (P14Entry, P14NamedMixin, P14AuditMixin, object)
try:
    P14Entry(name='x', audit_tag='y', unexpected=True)
except TypeError:
    pass
else:
    raise AssertionError('unexpected keyword should fail at object boundary')
print('cooperative constructor MRO:', [c.__name__ for c in P14Entry.__mro__])

cooperative constructor MRO: ['P14Entry', 'P14NamedMixin', 'P14AuditMixin', 'object']


**Why it works / takeaway:** `super()` means *next in the instance class's MRO*, not a hard-coded parent. Each mixin consumes its own keyword arguments, then forwards the remainder; `object.__init__` accepts no unexpected keyword arguments.

## Problem 15 — Initialize a diamond exactly once · Expert

**Learning goal:** Understand cooperative `super()` in multiple inheritance.

**Task:** Make a diamond hierarchy whose four constructors each log once. Confirm the actual MRO and avoid double-initializing the shared root.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [16]:
class P15Root:
    def __init__(self):
        self.events.append('root')
        super().__init__()

class P15Left(P15Root):
    def __init__(self):
        self.events.append('left')
        super().__init__()

class P15Right(P15Root):
    def __init__(self):
        self.events.append('right')
        super().__init__()

class P15Diamond(P15Left, P15Right):
    def __init__(self):
        self.events = ['diamond']
        super().__init__()

item = P15Diamond()
assert P15Diamond.__mro__ == (
    P15Diamond, P15Left, P15Right, P15Root, object
)
assert item.events == ['diamond', 'left', 'right', 'root']
assert item.events.count('root') == 1
print('constructor sequence:', ' → '.join(item.events))

constructor sequence: diamond → left → right → root


**Why it works / takeaway:** Python computes a C3 linearized MRO. Cooperative methods each call `super()` once, permitting the common root to be reached once rather than once through each branch.

## Problem 16 — Inspect bound and unbound method objects · Advanced

**Learning goal:** Identify descriptors that turn functions into bound methods.

**Task:** Check that a class method retrieved through an instance is bound to that instance, and show which original function it wraps.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [17]:
class P16Greeter:
    def greet(self, punctuation='!'):
        return f'hello{punctuation}'

greeter = P16Greeter()
bound = greeter.greet
function = P16Greeter.greet
assert isinstance(bound, types.MethodType)
assert isinstance(function, types.FunctionType)
assert bound.__self__ is greeter
assert bound.__func__ is function
assert function(greeter, '?') == bound('?') == 'hello?'
assert function is vars(P16Greeter)['greet']
print('bound method target:', bound.__func__.__name__)

bound method target: greet


**Why it works / takeaway:** A user-defined function stored in a class is a descriptor. Access via an instance yields a bound method that carries `__self__` and references its original function via `__func__`.

---
# Part III — Attribute lookup, object hooks and descriptors

## Problem 17 — Implement a safe attribute fallback · Expert

**Learning goal:** Customize `__getattribute__` without accidental recursion.

**Task:** Log accesses to real attributes and synthesize a fallback only when normal lookup raises `AttributeError`. Ensure that missing attributes still fail.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [18]:
class P17Settings:
    def __init__(self):
        object.__setattr__(self, '_events', [])
        self.theme = 'dark'

    def __getattribute__(self, name):
        if name != '_events':
            object.__getattribute__(self, '_events').append(name)
        return object.__getattribute__(self, name)

    def __getattr__(self, name):
        if name == 'locale':
            return 'en-US'
        raise AttributeError(name)

settings = P17Settings()
assert settings.theme == 'dark'
assert settings.locale == 'en-US'
assert settings._events == ['theme', 'locale']
try:
    settings.unknown
except AttributeError:
    pass
else:
    raise AssertionError('unknown attribute must remain unknown')
print('intercepted requests:', settings._events)

intercepted requests: ['theme', 'locale', 'unknown']


**Why it works / takeaway:** `__getattribute__` is called for regular attribute access. Delegate using `object.__getattribute__`, not `self.some_attr`, to avoid recursion. `__getattr__` is the fallback that runs after an `AttributeError`.

## Problem 18 — Prove special-method lookup bypasses the instance dictionary · Expert

**Learning goal:** Understand how built-in operations choose special methods.

**Task:** Assign an instance-level `__len__` function, then compare direct invocation with the built-in `len()`. Override at the class level to change `len()`.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [19]:
class P18Container:
    def __len__(self):
        return 2

c = P18Container()
c.__len__ = lambda: 99
assert c.__len__() == 99
assert len(c) == 2

class P18ChildContainer(P18Container):
    def __len__(self):
        return 5

assert len(P18ChildContainer()) == 5
assert P18Container.__len__(c) == 2
print('instance attribute:', c.__len__(), '| built-in:', len(c))

instance attribute: 99 | built-in: 2


**Why it works / takeaway:** Implicit special-method lookup for `len`, arithmetic, and many other operators generally looks on the *type*, not the instance's ordinary attribute dictionary. The explicit `c.__len__()` expression performs normal attribute lookup.

## Problem 19 — Compare slotted and dictionary-backed instances · Advanced

**Learning goal:** Use `__slots__` with realistic limitations.

**Task:** Create a slotted class, block arbitrary attributes, and show that subclassing a slotted class without another `__slots__` declaration ordinarily restores an instance dictionary.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [20]:
class P19Slotted:
    __slots__ = ('x',)

    def __init__(self, x):
        self.x = x

class P19Subclass(P19Slotted):
    pass

slot_obj = P19Slotted(10)
assert not hasattr(slot_obj, '__dict__')
try:
    slot_obj.extra = 'not allowed'
except AttributeError:
    pass
else:
    raise AssertionError('unexpected attribute allowed')
sub = P19Subclass(10)
sub.extra = 'allowed'
assert sub.__dict__ == {'extra': 'allowed'}
assert sub.x == 10
print('base has __dict__?', hasattr(slot_obj, '__dict__'))
print('child has __dict__?', hasattr(sub, '__dict__'))

base has __dict__? False
child has __dict__? True


**Why it works / takeaway:** Slots can remove an ordinary instance dictionary and constrain allowed attributes. A subclass without its own `__slots__` normally gets a dictionary, so slots are not automatically transitive as a storage restriction.

## Problem 20 — Validate writes and deletions centrally · Expert

**Learning goal:** Override `__setattr__` and `__delattr__` without recursion.

**Task:** Reject negative balances and protect an identifier from deletion. Exercise allowed and disallowed paths with assertions.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [21]:
class P20Ledger:
    def __init__(self, identifier, balance=0):
        self.identifier = identifier
        self.balance = balance

    def __setattr__(self, name, value):
        if name == 'balance' and (not isinstance(value, (int, float)) or value < 0):
            raise ValueError('balance must be nonnegative')
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        if name == 'identifier':
            raise AttributeError('identifier cannot be deleted')
        object.__delattr__(self, name)

ledger = P20Ledger('L-1', 10)
ledger.balance = 15
assert ledger.balance == 15
try:
    ledger.balance = -1
except ValueError:
    pass
else:
    raise AssertionError('negative balance accepted')
assert ledger.balance == 15
try:
    del ledger.identifier
except AttributeError:
    pass
else:
    raise AssertionError('protected attribute deleted')
assert ledger.identifier == 'L-1'
print('validated ledger balance:', ledger.balance)

validated ledger balance: 15


**Why it works / takeaway:** Delegate the actual operation to `object.__setattr__` or `object.__delattr__`. Reassigning with `self.attribute = ...` from inside `__setattr__` would recurse.

## Problem 21 — Build a reusable data descriptor · Expert

**Learning goal:** Use `__set_name__`, `__get__`, and `__set__` to enforce invariants.

**Task:** Create a positive-number descriptor used by two different fields. Show that it works on both the class and the instance, and that an instance dictionary cannot shadow it.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [22]:
class P21Positive:
    def __set_name__(self, owner, name):
        self.storage_name = f'_validated_{name}'

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
            raise ValueError('expected a positive number')
        instance.__dict__[self.storage_name] = value

class P21Rectangle:
    width = P21Positive()
    height = P21Positive()

    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

rect = P21Rectangle(3, 4)
assert rect.area == 12
assert isinstance(P21Rectangle.width, P21Positive)
rect.__dict__['width'] = -99  # an ordinary key cannot shadow the data descriptor
assert rect.width == 3
try:
    rect.height = 0
except ValueError:
    pass
else:
    raise AssertionError('invalid height accepted')
assert rect.height == 4
print('descriptor-backed instance state:', rect.__dict__)

descriptor-backed instance state: {'_validated_width': 3, '_validated_height': 4, 'width': -99}


**Why it works / takeaway:** A *data descriptor* defines `__get__` and `__set__` (or `__delete__`) and takes precedence over the same-named instance-dictionary entry. `__set_name__` tells the reusable descriptor what attribute it manages.

## Problem 22 — Register subclasses at class-creation time · Expert

**Learning goal:** Use `__init_subclass__` for safe, explicit registration.

**Task:** Build a base class that registers concrete subclasses by a supplied keyword, rejects duplicate keys, and does not accidentally register the base itself.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [23]:
class P22Command:
    registry = {}

    def __init_subclass__(cls, *, command=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if command is None:
            return
        if command in P22Command.registry:
            raise ValueError(f'duplicate command: {command}')
        P22Command.registry[command] = cls

class P22Create(P22Command, command='create'):
    pass

class P22Delete(P22Command, command='delete'):
    pass

assert P22Command.registry == {'create': P22Create, 'delete': P22Delete}
assert type(P22Command.registry['create']()) is P22Create
try:
    class P22Duplicate(P22Command, command='create'):
        pass
except ValueError as exc:
    assert 'duplicate' in str(exc)
else:
    raise AssertionError('duplicate registration accepted')
assert len(P22Command.registry) == 2
print('registered commands:', sorted(P22Command.registry))

registered commands: ['create', 'delete']


**Why it works / takeaway:** `__init_subclass__` runs when a subclass is created. Consume your custom class keyword, then call `super().__init_subclass__(**kwargs)` to preserve cooperation with other hooks.

## Problem 23 — Generate a class dynamically with `type` · Expert

**Learning goal:** Connect metaclasses to real runtime behavior.

**Task:** Build a new class by calling three-argument `type`, attach an instance method, and confirm it participates in ordinary inheritance.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [24]:
class P23Base:
    def label(self):
        return 'base'

def p23_compute(self, n):
    return self.factor * n

P23Dynamic = type(
    'P23Dynamic',
    (P23Base,),
    {'factor': 7, 'compute': p23_compute, '__doc__': 'Generated class.'},
)
instance = P23Dynamic()
assert type(instance) is P23Dynamic
assert type(P23Dynamic) is type
assert P23Dynamic.__bases__ == (P23Base,)
assert P23Dynamic.__mro__[-1] is object
assert instance.label() == 'base'
assert instance.compute(6) == 42
assert P23Dynamic.__doc__ == 'Generated class.'
print('dynamically generated:', P23Dynamic.__name__, instance.compute(6))

dynamically generated: P23Dynamic 42


**Why it works / takeaway:** `type(name, bases, namespace)` is a class factory; it constructs a class from its name, base classes, and attributes. An ordinary `class` statement ultimately uses compatible class-creation machinery.

---
# Part IV — Robust representation, copying and lifetime

## Problem 24 — Prevent infinite recursion in `__repr__` · Expert

**Learning goal:** Make debugging representations safe for cyclic graphs.

**Task:** Represent a singly linked node, connect it back to itself, and use a standard-library decorator so `repr(node)` terminates without manually keeping global state.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [25]:
class P24Link:
    def __init__(self, value, next_link=None):
        self.value = value
        self.next_link = next_link

    @recursive_repr(fillvalue='...')
    def __repr__(self):
        return f'P24Link({self.value!r}, next_link={self.next_link!r})'

link = P24Link('loop')
link.next_link = link
text = repr(link)
assert text == "P24Link('loop', next_link=...)"
print('cycle-safe repr:', text)

cycle-safe repr: P24Link('loop', next_link=...)


**Why it works / takeaway:** `reprlib.recursive_repr` detects recursive representation calls on the same object and substitutes a marker. It protects debugging output from obvious reference cycles.

## Problem 25 — Compare shallow and deep copies of a graph · Expert

**Learning goal:** Distinguish identity, aliasing and recursive data structure copying.

**Task:** Build two linked nodes including a self-cycle. Copy them both ways, and verify which nested objects and cycles are shared versus recreated.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [26]:
class P25Node:
    def __init__(self, value):
        self.value = value
        self.next = None

head = P25Node('head')
tail = P25Node('tail')
head.next = tail
tail.next = tail

shallow = copy.copy(head)
deep = copy.deepcopy(head)
assert shallow is not head
assert shallow.next is tail  # alias retained
assert deep is not head
assert deep.next is not tail  # new nested object
assert deep.next.next is deep.next  # original cycle preserved in new graph
assert deep.value == head.value
print('shallow shares nested node:', shallow.next is tail)
print('deep preserves copied cycle:', deep.next.next is deep.next)

shallow shares nested node: True
deep preserves copied cycle: True


**Why it works / takeaway:** Shallow copying creates a new outer object but retains references to nested objects. Deep copying memoizes visited objects to preserve the graph's sharing and cycles while creating independent copies.

## Problem 26 — Allow weak references on a slotted instance · Expert

**Learning goal:** Understand object lifetime and `__weakref__` with slots.

**Task:** Prove that an ordinary slotted class cannot be weak-referenced unless its layout permits it. Then create a weak-referenceable class and confirm the reference does not keep its object alive.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [27]:
class P26NoWeakref:
    __slots__ = ('value',)

class P26Weakrefable:
    __slots__ = ('value', '__weakref__')

try:
    weakref.ref(P26NoWeakref())
except TypeError:
    pass
else:
    raise AssertionError('weakref unexpectedly accepted')

tracked = P26Weakrefable()
tracked.value = 42
ref = weakref.ref(tracked)
assert ref() is tracked
assert ref().value == 42
del tracked
assert ref() is None  # no other strong references remain
print('weak reference after deletion:', ref())

weak reference after deletion: None


**Why it works / takeaway:** A slotted class needs a `__weakref__` slot (or another layout that supplies weakref support) to allow weak references. A `weakref.ref` is not a strong reference; this example depends only on an object whose reference count falls to zero once its last strong name is deleted.

---
# Part V — Integrated capstone and debugging challenge

## Problem 27 — Capstone: implement a production-minded `Vector2` value object · Capstone

**Learning goal:** Combine allocation, validation, immutable state, display, equality, hashing and operator dispatch.

**Task:** Implement a slotted 2D vector. Require finite, non-Boolean numeric components; normalize `-0.0`; make values hashable; support vector addition; provide useful `repr`/`str`; reject incompatible addition via `NotImplemented`; test dictionary/set invariants.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [28]:
class P27Vector2:
    __slots__ = ('_x', '_y')

    def __init__(self, x, y):
        for name, value in (('x', x), ('y', y)):
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError(f'{name} must be a real number, not bool')
            if not math.isfinite(value):
                raise ValueError(f'{name} must be finite')
        object.__setattr__(self, '_x', float(x) + 0.0)
        object.__setattr__(self, '_y', float(y) + 0.0)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __setattr__(self, name, value):
        raise AttributeError('P27Vector2 is immutable')

    def __delattr__(self, name):
        raise AttributeError('P27Vector2 is immutable')

    def __repr__(self):
        return f'P27Vector2({self.x!r}, {self.y!r})'

    def __str__(self):
        return f'<{self.x:g}, {self.y:g}>'

    def __eq__(self, other):
        if type(other) is type(self):
            return self.x == other.x and self.y == other.y
        return NotImplemented

    def __hash__(self):
        return hash((type(self), self.x, self.y))

    def __add__(self, other):
        if type(other) is type(self):
            return type(self)(self.x + other.x, self.y + other.y)
        return NotImplemented

    def __radd__(self, other):
        return self.__add__(other)

    def magnitude(self):
        return math.hypot(self.x, self.y)

v = P27Vector2(3, 4)
copy_v = P27Vector2(3.0, 4.0)
assert v is not copy_v and v == copy_v
assert hash(v) == hash(copy_v)
assert {v: 'found'}[copy_v] == 'found'
assert len({v, copy_v}) == 1
assert repr(v) == 'P27Vector2(3.0, 4.0)'
assert str(v) == '<3, 4>'
assert v.magnitude() == 5.0
assert v + P27Vector2(1, 2) == P27Vector2(4, 6)
assert sum([P27Vector2(1, 2), P27Vector2(3, 4)], P27Vector2(0, 0)) == P27Vector2(4, 6)
assert P27Vector2(-0.0, 0.0) == P27Vector2(0, -0.0)
assert P27Vector2.__add__(v, object()) is NotImplemented
for args, error in [((True, 1), TypeError), ((float('nan'), 1), ValueError),
                    ((float('inf'), 1), ValueError), (('3', 4), TypeError)]:
    try:
        P27Vector2(*args)
    except error:
        pass
    else:
        raise AssertionError(f'expected {error.__name__} for {args!r}')
try:
    v.x = 100
except AttributeError:
    pass
else:
    raise AssertionError('ordinary mutation accepted')
try:
    v + 'not a vector'
except TypeError:
    pass
else:
    raise AssertionError('incompatible addition should raise')
print('vector:', v, 'magnitude:', v.magnitude())

vector: <3, 4> magnitude: 5.0


**Why it works / takeaway:** This integrates many of the lesson's mechanisms. Only stable, equality-relevant components enter the hash; unsupported operands use `NotImplemented`; input checks exclude NaN and infinity; and `__repr__` conveys a reproducible-looking constructor form. In Python, explicit low-level bypasses can still defeat ordinary immutability, and very large integer-to-float conversion can raise `OverflowError`; address those cases if required by your API.

## Problem 28 — Debug a five-bug class in a regression suite · Capstone

**Learning goal:** Detect incorrect equality, mutable hashing, recursive attribute writes, weak validation, and ambiguous representations.

**Task:** Repair a record model using principles from previous problems. Ensure it rejects empty labels, compares by immutable contents, works as a dictionary key, and has an informative representation. Write tests that fail if each requirement regresses.

**Hint:** Trace the relevant special methods and predict the behavior before executing the solution.

**Solution (run the next cell):**

In [29]:
class P28Record:
    __slots__ = ('_label', '_revision')

    def __init__(self, label, revision):
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label must be a nonempty string')
        if type(revision) is not int or revision < 0:
            raise ValueError('revision must be a nonnegative integer')
        object.__setattr__(self, '_label', label)
        object.__setattr__(self, '_revision', revision)

    @property
    def label(self):
        return self._label

    @property
    def revision(self):
        return self._revision

    def __setattr__(self, name, value):
        raise AttributeError('P28Record cannot be mutated')

    def __delattr__(self, name):
        raise AttributeError('P28Record cannot be mutated')

    def __repr__(self):
        return f'P28Record(label={self.label!r}, revision={self.revision!r})'

    def __eq__(self, other):
        if type(other) is type(self):
            return (self.label, self.revision) == (other.label, other.revision)
        return NotImplemented

    def __hash__(self):
        return hash((type(self), self.label, self.revision))

a = P28Record('alpha', 2)
b = P28Record('alpha', 2)
c = P28Record('alpha', 3)
assert a == b and a is not b
assert a != c and a != object()
assert hash(a) == hash(b)
assert {a: 'ok'}[b] == 'ok'
assert repr(a) == "P28Record(label='alpha', revision=2)"
assert 'P28Record' in repr(a)
for invalid in [('', 0), ('  ', 0), (None, 0), ('x', -1), ('x', True)]:
    try:
        P28Record(*invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f'invalid record accepted: {invalid!r}')
for change in (lambda: setattr(a, 'label', 'beta'),
               lambda: setattr(a, '_revision', 9),
               lambda: delattr(a, '_label')):
    try:
        change()
    except AttributeError:
        pass
    else:
        raise AssertionError('unexpected mutation accepted')
assert {a: 'ok'}[a] == 'ok'
print('record regression checks: PASS')

record regression checks: PASS


**Why it works / takeaway:** A robust value object validates inputs once, stores fields with controlled ordinary access, defines equality over a stable tuple of fields, supplies a matching hash, and uses a precise `repr`. Assertions provide a lightweight regression suite.

---
# Consolidated review and additional independent challenges

Try these **without looking back**; use the earlier solutions as reference if needed.

1. Explain why `type(object) is type` and `type(type) is type` are both true.
2. Show a case where direct `object.__eq__` returns `NotImplemented` while the `==` expression evaluates to `False`.
3. Make a class with only `__repr__`, and predict the result of calling `str()`.
4. Explain why defining `__eq__` can make a class unhashable.
5. Implement a frozen three-dimensional key and verify `a == b → hash(a) == hash(b)` for your sample instances.
6. Identify when subclass equality receives priority over base-class equality.
7. Explain why `super()` should not be understood as a synonym for a named parent.
8. Show why assigning `instance.__len__` does not normally change `len(instance)`.
9. Explain descriptor precedence over `instance.__dict__`.
10. Explain why adding `__slots__` to a base class may not prevent attributes on a subclass.
11. State when `__new__` rather than `__init__` is necessary for an immutable built-in subclass.
12. Explain why a custom `__getattribute__` should delegate to `object.__getattribute__`.
13. Use a weak reference to observe object deletion without keeping the object alive.
14. Show the aliasing difference between `copy.copy` and `copy.deepcopy` on a cyclic graph.
15. Explain why a good debug `repr` should escape user strings and account for cycles.

**Checklist for reliable code:** Predict first; implement with small functions/classes; test equality reflexivity/symmetry for supported values, hash invariants, invalid input, inheritance behavior, special-method dispatch, and both typical and edge cases. Avoid asserting numerical `id()` values, fixed hashes, or default memory-address strings; these are runtime-specific. Hash collisions are possible, and deliberate low-level mutation can circumvent ordinary `__setattr__` protections.

In [30]:
# Final smoke test: representative guarantees from the entire notebook.
assert issubclass(P01Node, object)
assert object.__eq__(object(), object()) is NotImplemented
assert P09MutableValue.__hash__ is None
assert P10Point(1, 2) == P10Point(1, 2)
assert len(P18Container()) == 2
assert P21Rectangle(2, 3).area == 6
assert P27Vector2(3, 4).magnitude() == 5
assert {P28Record('done', 1): 'passed'}[P28Record('done', 1)] == 'passed'
print('ALL 28 PROBLEM SOLUTIONS AND FINAL SMOKE TEST COMPLETED')

ALL 28 PROBLEM SOLUTIONS AND FINAL SMOKE TEST COMPLETED
